# Filtering d_dose — disease-specific dosing test cases

Test cases exercising the disease-specific dose filtering logic used by the DOSAGE validator app (`main.py`). Demonstrates filtering by generic name, disease, age, and administration route.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the dataset
d_dose = pd.read_csv("d_dose.csv")

In [ ]:
# Create mapping of unique (generic, disease) pairs
generic_disease_map = d_dose[['generic', 'disease']].drop_duplicates().sort_values(['generic', 'disease'])

# Optional: export to CSV
generic_disease_map.to_csv("generic_disease_map.csv", index=False)

In [ ]:
def filter_d_dose(generic=None, disease=None, age=None, route=None, age_unit='y'):
    """
    Filter d_dose by generic, disease, age, and route.
    - age_unit: 'y' for years, 'm' for months, 'd' for days
    Returns a filtered DataFrame with all matching rows.
    """
    df = d_dose.copy()
    
    # Apply generic filter
    if generic:
        df = df[df['generic'].str.lower() == generic.lower()]
        
    # Apply disease filter
    if disease:
        df = df[df['disease'].str.lower() == disease.lower()]
        
    # Apply route filter
    if route:
        df = df[df['route'].str.upper() == route.upper()]
        
    # Handle age filtering
    if age is not None:
        age_y = age if age_unit == 'y' else age / 12 if age_unit == 'm' else age / 365
        age_m = age if age_unit == 'm' else age * 12 if age_unit == 'y' else age / 30.44
        age_d = age if age_unit == 'd' else age * 365 if age_unit == 'y' else age * 30.44

        def age_match(row):
            for min_col, max_col, val in [
                ('min_age_y', 'max_age_y', age_y),
                ('min_age_m', 'max_age_m', age_m),
                ('min_age_d', 'max_age_d', age_d),
            ]:
                if pd.notna(row[min_col]) and pd.notna(row[max_col]):
                    if float(row[min_col]) <= val <= float(row[max_col]):
                        return True
            return False
        
        df = df[df.apply(age_match, axis=1)]
    
    return df.reset_index(drop=True)

# ▶ Example usage:
# Filter for Ampicillin, Gastroenteritis, age 12 years, PO route
result = filter_d_dose(generic="Tinidazole", disease="Trichomoniasis", age=10, route="PO", age_unit='y')

# View all relevant fields
print(result)


      generic         disease  min_age_d  max_age_d  min_age_m  max_age_m  \
0  Tinidazole  Trichomoniasis        NaN        NaN        NaN        NaN   

   min_age_y  max_age_y  min_weight  max_weight  ...  min_dw_iu  max_dw_iu  \
0        3.0       11.0         NaN         NaN  ...        NaN        NaN   

   limit_mg  limit_iu  min_dd_mg  max_dd_mg  min_dd_iu  max_dd_iu  route  \
0       NaN       NaN        NaN        NaN        NaN        NaN     PO   

   Unnamed: 21  
0          NaN  
